# ArchForge — Colab Run Book

Mamluk / Islamic Cairo style adaptation with a FLUX.1-dev LoRA. Time-boxed, 5 hours total.

**Read this before running anything.** Three findings from pre-flight work change the plan in `PROJECT_BRIEF.md`:

1. **ai-toolkit is not viable on a T4.** Its own FAQ states a 24GB VRAM minimum, it ships no NF4 quantiser, and its default `qfloat8` is FP8 — not accelerated below sm_89. The brief lists ai-toolkit as *preferred* and diffusers as *fallback*; on this hardware that ordering is inverted.
2. **The stock diffusers FLUX DreamBooth LoRA script has no quantisation support at all** (verified against source — no `--quantization` flag exists). A 23.8GB fp16 transformer cannot train on 16GB. The usable path is `examples/research_projects/flux_lora_quantization/train_dreambooth_lora_flux_miniature.py`, which hardcodes NF4.
3. **Both scripts in that research project are hardcoded to the `Norod78/Yarn-art-style` demo dataset.** They load images from that HF dataset, not from disk, and the parquet stores only embeddings keyed by an image hash. `scripts/patch_diffusers.py` redirects them to a local dataset built from our 39 images.

**Hardware realities on a T4 (sm_75):** FP8 is impossible; bf16 is software-emulated and 2–4× slow, so training is **fp16 + GradScaler**. Flash Attention needs sm_80+ — do not install it; diffusers' default SDPA backend works. T5-XXL fp16 (9.5GB) + NF4 transformer (6.7GB) exceeds 16GB, so **text-embedding pre-caching is mandatory**.

## Phase 0 — Hardware check

Clones the repo and reports GPU, VRAM, CUDA, disk, HF auth, quantisation support and Drive status.

**Before running this cell:** open the Secrets panel (key icon, left sidebar), add a
secret named `HF_TOKEN` with your Hugging Face **write** token, and switch on notebook
access for it. Do not paste the token into a cell — the notebook is shareable and the
token would travel with it.

Your HF account also needs approved access to `black-forest-labs/FLUX.1-dev`; request
it at https://huggingface.co/black-forest-labs/FLUX.1-dev if you have not already.

**Read the output before continuing.** If `fp8 supported` is True on a non-Ada card, or VRAM is under ~15GB, fix the Phase 5 config before spending an hour on it.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Add HF_TOKEN in the Colab Secrets panel (key icon, left sidebar) and enable
# notebook access. Never paste the token into a cell - Colab notebooks are shareable.
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get('HF_TOKEN'))

import os
REPO = '/content/ArchForge'
if not os.path.exists(REPO):
    !git clone https://github.com/marwantosolve/ArchForge.git $REPO
%cd $REPO
!git pull --quiet

!python scripts/check_env.py

## Data

If `data/train` already has the 39 curated 512px images (uploaded as a zip to Drive), it unzips. Otherwise the pipeline rebuilds the dataset deterministically from Wikimedia Commons — same seed, same allowlist in `data/selected.csv`, same output. The rebuild takes ~5 minutes.

In [ ]:
import os, zipfile

TRAIN, VALIDATION = 33, 6


def dataset_ready():
    for split, expected in (('train', TRAIN), ('validation', VALIDATION)):
        if not os.path.isdir(f'data/{split}') or len(os.listdir(f'data/{split}')) != expected:
            return False
    return True


# Never glob recursively over the mounted Drive - it walks every directory over the
# network and hangs. Check exact paths only.
CANDIDATES = [
    'archforge_dataset.zip',
    '/content/ArchForge/archforge_dataset.zip',
    '/content/drive/MyDrive/archforge_dataset.zip',
    '/content/drive/MyDrive/archforge/archforge_dataset.zip',
]

if not dataset_ready():
    found = next((p for p in CANDIDATES if os.path.exists(p)), None)
    if found:
        with zipfile.ZipFile(found) as archive:
            archive.extractall('.')
        print('extracted', found)
    else:
        print('no zip at any known path - rebuilding from Wikimedia (~5 min)')
        print('checked:', CANDIDATES)
        !pip install --quiet requests pillow
        !python scripts/collect_dataset.py --per-category 8 --target 130
        !python scripts/build_dataset.py
else:
    print('dataset already present')

print()
!python -c "import os; print('train:', len(os.listdir('data/train'))); print('val:', len(os.listdir('data/validation')))"
!head -3 metadata.csv

## Phase 3 — Captioning

Florence-2 generates a base description per image; the structured style template is then appended, with the architectural vocabulary grounded in each image's recorded Wikimedia category — so the template cannot hallucinate a minaret onto a doorway.

Outputs `captions.jsonl` and fills the `caption` column of `metadata.csv`.

**Spot-check 5–10 captions for hallucinations.** That is the point of the review step in the brief.

In [ ]:
!pip install --quiet 'transformers<5' timm einops
!python scripts/caption_dataset.py

import json
for i, line in enumerate(open('captions.jsonl')):
    if i >= 8: break
    r = json.loads(line)
    print(f"--- {r['image_path']}")
    print(f"    {r['caption']}")
    print(f"    [vlm base: {r['vlm_base'][:90] or 'NONE - template only'}]")

## Phase 4 — Baseline generation

Six fixed prompts from `prompts.json`, base model only, no LoRA. **These exact prompts and seeds are reused unchanged in Phase 6** — do not edit `prompts.json` after this point.

Saved to `results/baseline/`. The manifest records seed, steps, guidance and resolution so the LoRA run can be held identical.

In [ ]:
!pip install --quiet diffusers transformers accelerate bitsandbytes peft safetensors

!python scripts/generate.py --out results/baseline --tag base

import json
m = json.load(open('results/baseline/manifest.json'))
print(f"{sum(x['seconds'] for x in m)/60:.1f} min total, {m[0]['seconds']:.1f}s per image")
print(f"steps={m[0]['steps']} guidance={m[0]['guidance_scale']} res={m[0]['width']}")

## Phase 5a — Training setup

Clones diffusers, patches the two hardcoded dataset references, builds a local HF dataset from our images + captions, and precomputes the T5 text embeddings.

**The embeddings step is separate on purpose.** T5-XXL at 9.5GB fp16 can spike Colab's ~12.7GB of *host RAM* and kill the session. Keeping it in its own cell means a crash here does not cost you the training run.

In [ ]:
%cd /content
!git clone --quiet --depth 1 https://github.com/huggingface/diffusers.git
%cd /content/ArchForge

!python scripts/patch_diffusers.py --repo /content/diffusers
!pip install --quiet datasets
!python scripts/build_hf_dataset.py

import os
os.environ['ARCHFORGE_DATASET'] = '/content/ArchForge/data/hf_dataset'

%cd /content/diffusers/examples/research_projects/flux_lora_quantization
!python compute_embeddings.py --output_path /content/ArchForge/embeddings.parquet
%cd /content/ArchForge

# The embeddings must cover OUR images. If the patch above ever fails, both demo
# scripts silently fall back to the Norod78 yarn-art dataset and the run trains on
# knitting photos - the images would still look plausible. Stop here if this trips.
import pandas as pd
from datasets import load_from_disk

df = pd.read_parquet('embeddings.parquet')
expected = len(load_from_disk('data/hf_dataset'))
print(f'embeddings: {df.shape}  local images: {expected}')

assert len(df) == expected, (
    f'EMBEDDINGS DO NOT MATCH THE DATASET ({len(df)} vs {expected}).\n'
    'The diffusers patch did not take effect. Do NOT train - it would train on the '
    'Norod78 yarn-art demo dataset. Re-run the patch cell and read its output.'
)
print('OK - embeddings cover the local dataset')

## Phase 5b — Smoke test (5 steps)

**Run this before the real training run.** It is cheap and catches the failures that would otherwise surface 40 minutes into a long run: a hash mismatch between the parquet and the dataset, a dtype error, an OOM, a bad argument name.

If this does not complete, do not start Phase 5c.

I could not execute this on real hardware while building it — no GPU was available. The smoke test exists precisely because of that.

In [ ]:
import os
os.environ['ARCHFORGE_DATASET'] = '/content/ArchForge/data/hf_dataset'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

%cd /content/diffusers/examples/research_projects/flux_lora_quantization

!accelerate launch train_dreambooth_lora_flux_miniature.py \
  --pretrained_model_name_or_path="black-forest-labs/FLUX.1-dev" \
  --data_df_path="/content/ArchForge/embeddings.parquet" \
  --output_dir="/content/ArchForge/smoke_test" \
  --mixed_precision="fp16" \
  --use_8bit_adam \
  --weighting_scheme="none" \
  --resolution=512 \
  --train_batch_size=1 \
  --repeats=1 \
  --learning_rate=1e-4 \
  --guidance_scale=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --cache_latents \
  --rank=8 \
  --max_train_steps=5 \
  --seed=0

%cd /content/ArchForge

## Phase 5c — Training

Output goes **straight to Drive**, so every checkpoint lands there automatically and a disconnect costs at most 100 steps.

Time budget: the reference run is 700 steps at 1024px on a 4090. On a T4 at 512px expect roughly 4–8s per step, so ~400 steps is 30–55 minutes. **Read the per-step timing in the first 50 steps and adjust `STEPS` before it runs long.** If the clock is tight, cut steps — never cut the evaluation.

Record the wall-clock time and the exact config; the report needs them.

In [ ]:
import os, time
os.environ['ARCHFORGE_DATASET'] = '/content/ArchForge/data/hf_dataset'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

OUT = '/content/drive/MyDrive/archforge/models/archforge_lora'
os.makedirs(OUT, exist_ok=True)

STEPS = 400  # adjust from the smoke-test timing before running

%cd /content/diffusers/examples/research_projects/flux_lora_quantization

start = time.time()
!accelerate launch train_dreambooth_lora_flux_miniature.py \
  --pretrained_model_name_or_path="black-forest-labs/FLUX.1-dev" \
  --data_df_path="/content/ArchForge/embeddings.parquet" \
  --output_dir="$OUT" \
  --mixed_precision="fp16" \
  --use_8bit_adam \
  --weighting_scheme="none" \
  --resolution=512 \
  --train_batch_size=1 \
  --repeats=1 \
  --learning_rate=1e-4 \
  --guidance_scale=1 \
  --gradient_accumulation_steps=4 \
  --gradient_checkpointing \
  --lr_scheduler="constant" \
  --lr_warmup_steps=0 \
  --cache_latents \
  --rank=8 \
  --max_train_steps=$STEPS \
  --checkpointing_steps=100 \
  --seed=0
print(f'wall clock: {(time.time()-start)/60:.1f} min')

%cd /content/ArchForge
!ls -la "$OUT"

## Phase 6 — LoRA generation

Same six prompts, same seeds, same steps/guidance/resolution as Phase 4. Only the LoRA differs — that is what makes the comparison controlled.

If the LoRA run used a different step count or resolution than the baseline, the comparison is not valid. Check the manifests match before reading anything into the images.

In [ ]:
!python scripts/generate.py \
  --out results/comparison \
  --tag lora \
  --lora /content/drive/MyDrive/archforge/models/archforge_lora

import json
b = json.load(open('results/baseline/manifest.json'))
l = json.load(open('results/comparison/manifest.json'))
keys = ['steps', 'guidance_scale', 'height', 'width', 'seed']
mismatch = [(k, b[i][k], l[i][k]) for i in range(len(b)) for k in keys if b[i][k] != l[i][k]]
print('CONTROLLED - no differences' if not mismatch else f'MISMATCH: {mismatch}')

## Phase 7 — Comparison grid

Prompt | Base | LoRA, all six pairs. Saved to `reports/comparison_grid.png`.

Judge each pair on style fidelity, structural plausibility, style bleeding, hallucination and prompt adherence. **Write the failures down while you are looking at them** — the report needs 1–2 honest ones.

In [ ]:
!python scripts/make_grid.py \
  --baseline results/baseline \
  --lora results/comparison \
  --out reports/comparison_grid.png

from IPython.display import Image as IPImage, display
display(IPImage('reports/comparison_grid.png'))

import shutil, os
dst = '/content/drive/MyDrive/archforge/results'
os.makedirs(dst, exist_ok=True)
for d in ('results/baseline', 'results/comparison', 'reports'):
    if os.path.isdir(d):
        shutil.copytree(d, os.path.join(dst, os.path.basename(d)), dirs_exist_ok=True)
print('results copied to Drive')

## Phase 8 — Publish the results

Generates a model card for the adapter and an evaluation card, then pushes both to
the Hub. The LoRA goes to a model repo; the grid, both result sets and the
evaluation write-up go to a second model repo so the evaluation has a stable link.

**Set `HF_USER` and confirm the repo names before running.** Checkpoint folders are
excluded — only the final adapter weights are uploaded.

This produces the last two links the email needs. The dataset link comes from
`scripts/publish_dataset.py`, run at the end of Phase 3 or locally.

In [ ]:
import os

HF_USER = 'marwantosolve'  # change if your Hugging Face username differs
LORA_REPO = f'{HF_USER}/archforge-mamluk-cairo-lora'
RESULTS_REPO = f'{HF_USER}/archforge-mamluk-cairo-results'
DATASET_REPO = f'{HF_USER}/archforge-mamluk-cairo'
OUT = '/content/drive/MyDrive/archforge/models/archforge_lora'

!pip install --quiet huggingface_hub datasets

!python scripts/publish_dataset.py --repo-id "$DATASET_REPO" --push

!python scripts/publish_results.py \
  --lora-dir "$OUT" \
  --lora-repo "$LORA_REPO" \
  --results-repo "$RESULTS_REPO" \
  --dataset-repo "$DATASET_REPO" \
  --push

print()
print('DATASET  https://huggingface.co/datasets/' + DATASET_REPO)
print('LORA     https://huggingface.co/' + LORA_REPO)
print('RESULTS  https://huggingface.co/' + RESULTS_REPO)
print('REPO     https://github.com/marwantosolve/ArchForge')

## Before you close the session

Fill in `reports/evaluation.md` while the results are in front of you. It must contain at least one honest failure case — that is what makes this credible work rather than a demo.

Record: base model, rank, steps, LR, optimizer, resolution, hardware, wall-clock duration, and how many images actually reached training.